In [1]:
import sys
METRICS_BASE_PATH = '../../experiments'
sys.path.insert(0, METRICS_BASE_PATH)

In [2]:
import asyncio

from metrics.my_ragas import RagasMetricsConfig, RagasMetrics
from metrics.llm_as_a_judge_mine import MINEJudge, MINEJudgeConfig
from metrics.llm_as_a_judge import AnswersJudge, AnswersJudgeConfig
from metrics.base_metrics import ReaderMetrics

/home/dzigen/Desktop/Projects/PersonalAI/.pai_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package wordnet to /home/dzigen/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /home/dzigen/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
# верный ответ
example1 = {
    'user_input': "Когда родился Фёдор Михайловчи Достоевский?",
    'reference': "11 ноября 1821 г.",
    'response': "11 ноября 1821 г.",
    'retrieved_contexts': ["Пушкин родился 06.06.1799", "Максим Горький родился 28.03.1868", "Достоевский родился 11.11.1821"] 
}

# неверный ответ
example2 = {
    'user_input': "Когда родился Лев Николаевич Толстой?",
    'reference': "9 сентября 1828 г.",
    'response': "11 ноября 1821 г.",
    'retrieved_contexts': ["Пушкин родился 06.06.1799", "Максим Горький родился 28.03.1868", "Достоевский родился 11.11.1821"] 
}

# ответ по смыслу неверный
example3 = {
    'user_input': "У лошади четыре ноги?",
    'reference': "Да",
    'response': "У лошади одна передняя нога и две заднии.",
    'retrieved_contexts': [
        "Передние ноги лошади обычно выдерживают большую часть веса, а задние обеспечивают движение."
    ] 
}

# ответ по смыслу верный
example4 = {
    'user_input': "У лошади четрые ноги?",
    'reference': "Да",
    'response': "У лошади две переднии ноги и две заднии.",
    'retrieved_contexts': [
        "Лощадь это четвероногое животное, конечности которого заканчиваются копытами.", 
        "Передние ноги лошади обычно выдерживают большую часть веса, а задние обеспечивают движение."
    ] 
}


examples = [example1, example2, example3, example4]

##### Checking ragas metrics

In [10]:
ragas_config = RagasMetricsConfig()
ragas_main = RagasMetrics(ragas_config)

In [14]:
ragas_mname = ['response_groundedness', 'context_relevance', 'faithfulness', 'context_entity_recall']
for i, example in enumerate(examples):
    print(f"Example #{i}:\n{example}")
    for metric_name in ragas_mname:
        score = await ragas_main.perform(metric_name, **example)
        print(f"- {metric_name}: {score}")

Example #0:
{'user_input': 'Когда родился Фёдор Михайловчи Достоевский?', 'reference': '11 ноября 1821\u202fг.', 'response': '11 ноября 1821\u202fг.', 'retrieved_contexts': ['Пушкин родился 06.06.1799', 'Максим Горький родился 28.03.1868', 'Достоевский родился 11.11.1821']}
- response_groundedness: 1.0
- context_relevance: 1.0
- faithfulness: 1.0
- context_entity_recall: 0.0
Example #1:
{'user_input': 'Когда родился Лев Николаевич Толстой?', 'reference': '9 сентября 1828\u202fг.', 'response': '11 ноября 1821\u202fг.', 'retrieved_contexts': ['Пушкин родился 06.06.1799', 'Максим Горький родился 28.03.1868', 'Достоевский родился 11.11.1821']}
- response_groundedness: 1.0
- context_relevance: 0.0
- faithfulness: 0.0
- context_entity_recall: 0.0
Example #2:
{'user_input': 'У лошади четыре ноги?', 'reference': 'Да', 'response': 'У лошади одна передняя нога и две заднии.', 'retrieved_contexts': ['Передние ноги лошади обычно выдерживают большую часть веса, а задние обеспечивают движение.']}
- 

##### Checking LLM-as-a-Judge

In [6]:
judge_config = AnswersJudgeConfig()
judge_main = AnswersJudge(judge_config)

In [7]:
for i, example in enumerate(examples):
    print(f"Example #{i}:\n{example}")
    score = judge_main.perform(example['user_input'], example['reference'], example['response'])
    print(f"- LLM-as-a-Judge: {score}")

Example #0:
{'user_input': 'Когда родился Фёдор Михайловчи Достоевский?', 'reference': '11 ноября 1821\u202fг.', 'response': '11 ноября 1821\u202fг.', 'retrieved_contexts': ['Пушкин родился 06.06.1799', 'Максим Горький родился 28.03.1868', 'Достоевский родился 11.11.1821']}
- LLM-as-a-Judge: (1, ReturnInfo(occurred_warning=[], status=<ReturnStatus.success: 0>, message=''))
Example #1:
{'user_input': 'Когда родился Лев Николаевич Толстой?', 'reference': '9 сентября 1828\u202fг.', 'response': '11 ноября 1821\u202fг.', 'retrieved_contexts': ['Пушкин родился 06.06.1799', 'Максим Горький родился 28.03.1868', 'Достоевский родился 11.11.1821']}
- LLM-as-a-Judge: (0, ReturnInfo(occurred_warning=[], status=<ReturnStatus.success: 0>, message=''))
Example #2:
{'user_input': 'У лошади четыре ноги?', 'reference': 'Да', 'response': 'У лошади одна передняя нога и две заднии.', 'retrieved_contexts': ['Передние ноги лошади обычно выдерживают большую часть веса, а задние обеспечивают движение.']}
- LLM-

##### Checking MINE LLM-as-a-Judge

In [8]:
minejudge_config = MINEJudgeConfig()
minejudge_main = MINEJudge(minejudge_config)

In [9]:
for i, example in enumerate(examples):
    print(f"Example #{i}:\n{example}")
    score = minejudge_main.perform(example['user_input'], example['retrieved_contexts'])
    print(f"- MINE LLM-as-a-Judge: {score}")

Example #0:
{'user_input': 'Когда родился Фёдор Михайловчи Достоевский?', 'reference': '11 ноября 1821\u202fг.', 'response': '11 ноября 1821\u202fг.', 'retrieved_contexts': ['Пушкин родился 06.06.1799', 'Максим Горький родился 28.03.1868', 'Достоевский родился 11.11.1821']}
- MINE LLM-as-a-Judge: 1
Example #1:
{'user_input': 'Когда родился Лев Николаевич Толстой?', 'reference': '9 сентября 1828\u202fг.', 'response': '11 ноября 1821\u202fг.', 'retrieved_contexts': ['Пушкин родился 06.06.1799', 'Максим Горький родился 28.03.1868', 'Достоевский родился 11.11.1821']}
- MINE LLM-as-a-Judge: 0
Example #2:
{'user_input': 'У лошади четыре ноги?', 'reference': 'Да', 'response': 'У лошади одна передняя нога и две заднии.', 'retrieved_contexts': ['Передние ноги лошади обычно выдерживают большую часть веса, а задние обеспечивают движение.']}
- MINE LLM-as-a-Judge: 0
Example #3:
{'user_input': 'У лошади четрые ноги?', 'reference': 'Да', 'response': 'У лошади две переднии ноги и две заднии.', 'retri

##### Checking base metrics

In [4]:
BERTSCORE_MODEL_PATH = "../../models/google/electra-base-discriminator"
METEOR_FILEP = f"{METRICS_BASE_PATH}/metrics/meteor"
EXACTMATCH_FILEP = f"{METRICS_BASE_PATH}/metrics/exact_match"

basemetrics_main = ReaderMetrics(BERTSCORE_MODEL_PATH, METEOR_FILEP, EXACTMATCH_FILEP)

Loading Meteor...
Loading ExactMatch
Loading BertScore


In [5]:
AVAILABLE_BASEMETRICS = [("f1", basemetrics_main.f1), ("levenshtain",basemetrics_main.levenshtain_score), 
                         ("exact_match", basemetrics_main.exact_match), ("meteor", basemetrics_main.meteor), 
                         ("bleu2", basemetrics_main.bleu2), ("bleu1", basemetrics_main.bleu1), 
                         ("rougel", basemetrics_main.rougel), ("bertscore", basemetrics_main.bertscore)]

for i, example in enumerate(examples):
    print(f"Example #{i}:\n{example}")
    for metric_name, metric_func in AVAILABLE_BASEMETRICS:
        score = metric_func([example['response']], [example['reference']])
        print(f"- {metric_name}: {score}")

Example #0:
{'user_input': 'Когда родился Фёдор Михайловчи Достоевский?', 'reference': '11 ноября 1821\u202fг.', 'response': '11 ноября 1821\u202fг.', 'retrieved_contexts': ['Пушкин родился 06.06.1799', 'Максим Горький родился 28.03.1868', 'Достоевский родился 11.11.1821']}
- f1: [1.0]
- levenshtain: [0]


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 140.34it/s]


- exact_match: [1.0]


100%|██████████| 1/1 [00:00<00:00, 207.25it/s]


...

- bleu1: [0.0]


100%|██████████| 1/1 [00:00<00:00, 433.52it/s]


- rougel: [0.0]


The following layers were not sharded: embeddings.LayerNorm.bias, embeddings.position_embeddings.weight, embeddings.token_type_embeddings.weight, encoder.layer.*.attention.self.key.weight, encoder.layer.*.intermediate.dense.weight, encoder.layer.*.attention.output.LayerNorm.weight, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.output.dense.bias, encoder.layer.*.attention.self.query.bias, encoder.layer.*.attention.self.query.weight, encoder.layer.*.output.LayerNorm.weight, encoder.layer.*.attention.output.LayerNorm.bias, embeddings.word_embeddings.weight, encoder.layer.*.output.LayerNorm.bias, encoder.layer.*.attention.self.value.weight, encoder.layer.*.attention.self.value.bias, encoder.layer.*.attention.output.dense.weight, encoder.layer.*.attention.output.dense.bias, encoder.layer.*.output.dense.weight, embeddings.LayerNorm.weight, encoder.layer.*.attention.self.key.bias


- bertscore: {'precision': 0.61068, 'recall': 0.68842, 'f1': 0.64722, 'hash': '../../models/google/electra-base-discriminator_LNone_no-idf'}
